# 허깅페이스 임베딩(HuggingFace Embeddings) (2026 업데이트판)

LangChain에서 Hugging Face 임베딩 모델을 쓰는 방법은 세 가지입니다.
1. **로컬 실행**: `HuggingFaceEmbeddings` (내부적으로 Sentence Transformers 사용)
2. **호스팅 추론**: `HuggingFaceEndpointEmbeddings` (Inference Providers / Inference Endpoints)
3. **자체 서버 운영**: TEI(Text Embeddings Inference) 서버 + `OpenAIEmbeddings` (OpenAI 호환 API)

세 방법 모두 `langchain-huggingface`(또는 `langchain-openai`)의 같은 `Embeddings` 인터페이스를 제공하므로, 로컬에서 시작해 나중에 서버로 옮겨도 나머지 코드는 바뀌지 않습니다.

### 원본 대비 변경 사항
| 항목 | 원본 | 현재 권장 |
|---|---|---|
| 토큰 전달 | `huggingfacehub_api_token=os.environ[...]` | 환경 변수 `HF_TOKEN`(또는 `HUGGINGFACEHUB_API_TOKEN`)만 설정하면 자동 인식 |
| 호스팅 추론 경로 | 모델만 지정 | `provider=` 로 Inference Provider 선택 가능 |
| 장치 지정 | `{"device": "mps"}` 하드코딩 | 생략 시 cuda → mps → cpu 자동 선택 |
| 쿼리/문서 프롬프트 | 사용 안 함 | `encode_kwargs` / `query_encode_kwargs` 의 `prompt` 로 모델 권장 접두어 적용 |
| 유사도 | 내적 | 정규화된 코사인 유사도 헬퍼 |
| FlagEmbedding | 모델을 세 번 로드 | 한 번 로드해 dense/sparse/ColBERT 동시 출력 |

In [ ]:
%pip install -qU langchain-huggingface sentence-transformers FlagEmbedding python-dotenv numpy

## 환경 설정

- `.env` 파일의 API 키를 `python-dotenv`로 불러옵니다.
- **변경점**: 책에서 사용한 `langchain_teddynote.logging.langsmith()`는 서드파티 헬퍼입니다. 현재 LangSmith 공식 방식은 환경 변수(`LANGSMITH_TRACING`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT`)만 설정하는 것이며, 별도 패키지가 필요 없습니다.
- 참고: 임베딩 호출(`embed_query`, `embed_documents`)은 Runnable이 아니어서 LangSmith에 트레이스가 남지 않습니다. 이 챕터에서는 없어도 되는 설정이지만, 이후 체인/에이전트 실습과 형태를 맞추기 위해 둡니다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # .env 파일의 키를 환경 변수로 로드

# LangSmith 추적 (LANGSMITH_API_KEY 는 .env 에 넣어 둡니다)
os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "CH08-Embeddings")

In [ ]:
import os

# ./cache/ 경로에 모델을 다운로드하도록 설정 (모델 로드 전에 설정해야 적용됩니다)
os.environ["HF_HOME"] = "./cache/"

## 샘플 데이터

In [ ]:
texts = [
    "안녕, 만나서 반가워.",
    "LangChain simplifies the process of building applications with large language models",
    "랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다. ",
    "LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.",
    "Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.",
]

In [ ]:
query = "LangChain 에 대해서 알려주세요."

**참고(Reference)**

![](./images/top-ranked-embeddings.png)

- [(출처) Kor-IR: 한국어 검색을 위한 임베딩 벤치마크](https://github.com/teddylee777/Kor-IR?tab=readme-ov-file)
- 최신 순위는 [MTEB 리더보드](https://huggingface.co/spaces/mteb/leaderboard)에서 확인할 수 있습니다.

유사도 계산 결과를 출력합니다.

**변경점**: 원본은 `query @ documents.T`(내적)만 사용했습니다. 내적은 벡터가 **정규화되어 있을 때만** 코사인 유사도와 같습니다. 모델마다 정규화 여부가 다르므로, 아래처럼 명시적으로 정규화한 뒤 코사인 유사도를 계산하는 헬퍼를 쓰는 편이 안전합니다.

In [ ]:
import numpy as np


def cosine_scores(query_vec, doc_vecs):
    q = np.asarray(query_vec, dtype=np.float32)
    d = np.asarray(doc_vecs, dtype=np.float32)
    q = q / np.linalg.norm(q)
    d = d / np.linalg.norm(d, axis=1, keepdims=True)
    return d @ q


def print_ranking(query, query_vec, doc_vecs, docs):
    scores = cosine_scores(query_vec, doc_vecs)
    print(f"[Query] {query}\n" + "=" * 40)
    for rank, idx in enumerate(scores.argsort()[::-1]):
        print(f"[{rank}] 유사도: {scores[idx]:.3f} | {docs[idx]}\n")

## HuggingFace Endpoint Embedding

`HuggingFaceEndpointEmbeddings`는 내부적으로 `huggingface_hub`의 `InferenceClient`를 사용해 원격에서 임베딩을 계산합니다. 모델을 내려받지 않아도 됩니다.

**변경점**
- 토큰은 `.env`에 `HF_TOKEN=hf_...`(또는 `HUGGINGFACEHUB_API_TOKEN`)으로 넣어 두면 자동으로 읽습니다. 코드에 토큰을 직접 넘기지 않습니다.
- `provider=`로 추론을 처리할 Inference Provider를 고를 수 있습니다. 어떤 모델이 어떤 Provider에서 지원되는지는 [Inference Providers 문서](https://huggingface.co/docs/inference-providers)에서 확인하세요. 원본의 `intfloat/multilingual-e5-large-instruct`는 서버리스 제공 여부가 수시로 바뀌므로, 여기서는 LangChain 공식 문서 예시에 나오는 `BAAI/bge-m3` + `hf-inference` 조합을 사용합니다.
- `task="feature-extraction"`은 기본값이므로 생략합니다.

In [ ]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings

endpoint_model = "BAAI/bge-m3"

hf_endpoint_embeddings = HuggingFaceEndpointEmbeddings(
    model=endpoint_model,
    provider="hf-inference",
)

In [ ]:
%%time
embedded_documents = hf_endpoint_embeddings.embed_documents(texts)

In [ ]:
print("[HuggingFace Endpoint Embedding]")
print(f"Model: \t\t{endpoint_model}")
print(f"Dimension: \t{len(embedded_documents[0])}")

In [ ]:
embedded_query = hf_endpoint_embeddings.embed_query(query)
print_ranking(query, embedded_query, embedded_documents, texts)

### 유사도 계산의 수학적 의미

**벡터 내적**: $\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^{n} a_i b_i = \|\mathbf{a}\| \|\mathbf{b}\| \cos \theta$

**코사인 유사도**: $\cos \theta = \dfrac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|}$

내적은 벡터 크기(노름) $\|\mathbf{a}\| = \sqrt{a_1^2 + \cdots + a_n^2}$ 의 영향을 받습니다. 벡터를 길이 1로 정규화하면 $\|\mathbf{a}\| = \|\mathbf{b}\| = 1$ 이 되어 **내적 = 코사인 유사도**가 됩니다. 그래서 로컬 모델에서는 `normalize_embeddings=True`를 주고, 위의 `cosine_scores` 헬퍼는 어떤 모델이든 안전하도록 직접 정규화합니다.

## HuggingFace Embeddings (로컬 실행)

### `intfloat/multilingual-e5-large-instruct`

- [intfloat/multilingual-e5-large-instruct](https://huggingface.co/intfloat/multilingual-e5-large-instruct)

**변경점 — 쿼리/문서 프롬프트**

E5, BGE, Qwen3-Embedding 같은 최신 모델은 학습할 때 쿼리와 문서에 서로 다른 접두어를 붙였습니다. 모델 카드가 권장하는 접두어를 쓰지 않으면 검색 품질이 떨어집니다. `HuggingFaceEmbeddings`는 다음 두 인자로 이를 분리해 지정할 수 있습니다.
- `encode_kwargs`: `embed_documents()`에 적용
- `query_encode_kwargs`: `embed_query()`에 적용

`multilingual-e5-large-instruct`는 **쿼리에만** `Instruct: {작업 설명}\nQuery: ` 접두어를 붙이고, 문서에는 붙이지 않습니다. (`multilingual-e5-large`처럼 instruct가 아닌 E5는 `"query: "` / `"passage: "`를 사용합니다.)

**변경점 — 장치 지정**: `model_kwargs={"device": ...}`를 생략하면 Sentence Transformers가 cuda → mps → cpu 순서로 자동 선택합니다. 특정 장치를 강제하고 싶을 때만 지정하세요.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

e5_model = "intfloat/multilingual-e5-large-instruct"
e5_task = "Given a web search query, retrieve relevant passages that answer the query"

e5_embeddings = HuggingFaceEmbeddings(
    model_name=e5_model,
    # model_kwargs={"device": "cuda"},  # 필요할 때만 지정 (cuda / mps / cpu)
    encode_kwargs={"normalize_embeddings": True},
    query_encode_kwargs={
        "normalize_embeddings": True,
        "prompt": f"Instruct: {e5_task}\nQuery: ",
    },
)

In [ ]:
%%time
e5_documents = e5_embeddings.embed_documents(texts)

In [ ]:
print(f"Model: \t\t{e5_model}")
print(f"Dimension: \t{len(e5_documents[0])}")

In [ ]:
e5_query = e5_embeddings.embed_query(query)
print_ranking(query, e5_query, e5_documents, texts)

## BGE-M3 임베딩

`BAAI/bge-m3`는 접두어 없이 사용하는 다국어 모델입니다(최대 8192 토큰). 장치 옵션은 필요할 때만 지정합니다.
- `{"device": "mps"}`: Apple Silicon GPU (Mac)
- `{"device": "cuda"}`: NVIDIA GPU (CUDA 설치 필요)
- `{"device": "cpu"}`: 모든 환경

In [ ]:
bge_model = "BAAI/bge-m3"

bge_embeddings = HuggingFaceEmbeddings(
    model_name=bge_model,
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
)

In [ ]:
%%time
bge_documents = bge_embeddings.embed_documents(texts)

In [ ]:
print(f"Model: \t\t{bge_model}")
print(f"Dimension: \t{len(bge_documents[0])}")

bge_query = bge_embeddings.embed_query(query)
print_ranking(query, bge_query, bge_documents, texts)

### `FlagEmbedding`을 활용하는 방식

**참고**
- [FlagEmbedding - BGE-M3](https://github.com/FlagOpen/FlagEmbedding/tree/master/research/BGE_M3)

`FlagEmbedding`의 `BGEM3FlagModel`은 BGE-M3가 가진 세 가지 표현을 모두 꺼낼 수 있습니다. (LangChain의 `HuggingFaceEmbeddings`는 dense 벡터만 제공합니다.)

- **Dense vector**: 문장 전체의 의미를 담은 단일 벡터
- **Sparse (lexical weight)**: 토큰별 가중치로 정확한 단어 매칭
- **Multi-vector (ColBERT)**: 토큰별 벡터로 세밀한 문맥 매칭

**변경점**: 원본은 같은 모델을 세 번 로드했습니다(메모리·시간 낭비). 한 번 로드하고 `encode()`에서 세 가지 출력을 동시에 요청합니다.

In [ ]:
from FlagEmbedding import BGEM3FlagModel

# use_fp16=True: 약간의 정확도 손실과 함께 속도 향상 (GPU 권장)
bge_flagmodel = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)

bge_encoded = bge_flagmodel.encode(
    texts,
    batch_size=12,
    max_length=8192,  # 긴 입력이 필요 없으면 줄여서 속도 향상
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=True,
)

In [ ]:
# Dense 벡터 (행: 문장 수, 열: 차원)
print(f"Model: \t\tBAAI/bge-m3")
print(f"Dense shape: \t{bge_encoded['dense_vecs'].shape}")

### Sparse Embedding (Lexical Weight)

벡터의 대부분이 0인 고차원 표현입니다. BGE-M3는 각 토큰의 중요도(lexical weight)를 모델이 직접 예측합니다(BM25 같은 통계식이 아니라 학습된 가중치).

- 특정 단어·고유명사를 정확히 매칭하는 데 강합니다.
- Dense 검색과 결합하면(하이브리드 검색) 성능이 좋아집니다.

In [ ]:
lw = bge_encoded["lexical_weights"]

print(bge_flagmodel.compute_lexical_matching_score(lw[0], lw[0]))  # 0 <-> 0
print(bge_flagmodel.compute_lexical_matching_score(lw[0], lw[1]))  # 0 <-> 1

In [ ]:
# 토큰 id 대신 실제 토큰으로 가중치 확인
bge_flagmodel.convert_id_to_token(lw[:1])

### Multi-Vector (ColBERT)

문서와 쿼리의 **각 토큰마다** 벡터를 만들고, 쿼리 토큰별로 가장 비슷한 문서 토큰을 찾아 점수를 합산합니다(late interaction).

- 토큰 수준의 세밀한 매칭이 가능하고 긴 문서에 강합니다.
- 대신 저장 공간이 문장당 벡터 1개가 아니라 토큰 수만큼 필요합니다.

In [ ]:
cv = bge_encoded["colbert_vecs"]

print(bge_flagmodel.colbert_score(cv[0], cv[0]))  # 0 <-> 0
print(bge_flagmodel.colbert_score(cv[0], cv[1]))  # 0 <-> 1